In [9]:
import pandas as pd
df = pd.read_csv(r"C:\Users\DELL\Downloads\Customer Behaviour Analysis p\customer_shopping_data.xlsx.csv")

In [10]:
df.head()

,Customer ID,Age,Age Group,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Old,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Young,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Middle Aged,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Young,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Middle Aged,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Age Group               3900 non-null   str    
 3   Gender                  3900 non-null   str    
 4   Item Purchased          3900 non-null   str    
 5   Category                3900 non-null   str    
 6   Purchase Amount (USD)   3900 non-null   int64  
 7   Location                3900 non-null   str    
 8   Size                    3900 non-null   str    
 9   Color                   3900 non-null   str    
 10  Season                  3900 non-null   str    
 11  Review Rating           3863 non-null   float64
 12  Subscription Status     3900 non-null   str    
 13  Shipping Type           3900 non-null   str    
 14  Discount Applied        3900 non-null   str    
 15

In [12]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [13]:
df.isnull().sum()

Customer ID                0
Age                        0
Age Group                  0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [14]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x:x.fillna(x.median()))

In [15]:
df.isnull().sum()

Customer ID               0
Age                       0
Age Group                 0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [29]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df=df.rename(columns = {'purchase_amount_(usd)':'purchase_amount'})

In [30]:
df.columns

Index(['customer_id', 'age', 'age_group', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [33]:
df['frequency_of_purchases'] = df['frequency_of_purchases'].replace({
    'Bi-Weekly':'Fortnightly',
    'Every 3 Months':'Quarterly'
})

In [34]:
frequency_mapping = {
    'Weekly':7,
    'Fortnightly':14,
    'Monthly':30,
    'Quarterly':90,
    'Annually':365
}
df['frequently_purchased_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [35]:
df[['frequently_purchased_days','frequency_of_purchases']].head(10)

,frequently_purchased_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [38]:
df=df.drop('promo_code_used',axis=1)

In [39]:
df.columns

Index(['customer_id', 'age', 'age_group', 'gender', 'item_purchased',
       'category', 'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'frequently_purchased_days'],
      dtype='str')

In [41]:
pip install psycopg2-binary sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [42]:
from sqlalchemy import create_engine

USER = "postgres"
PASSWORD = "pass"
HOST = "localhost"
PORT = "5432"
DB_NAME = "customer_behaviour"

connection_string = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(connection_string)

table_name = "customer"
df.to_sql(table_name,engine, if_exists="replace", index=False)

print("Data imported successfully!")


Data imported successfully!
